In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
from pathlib import Path
import torch.nn as nn
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch.optim as optim
from collections import defaultdict
import random
import math

from torch import amp

In [2]:
if torch.cuda.is_available():
    print("Cuda ok")

Cuda ok


# Globals

In [3]:
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_SAMPLES= 15_000
TRAIN_SIZE = 0.8
BATCH_SIZE = 200

EMBEDDING_DIM = 256
EMBEDDING_FOURIER_DIM = 64
EMBEDDING_RGB_DIM = 64
WEIGHT_TASK = 0.5
LEARNING_RATE = 1e-4
EPOCHS = 50
DEVICE = "cuda"

IMAGE_SIZE = (224, 224)
Image.MAX_IMAGE_PIXELS = None

# Utils

## Early Stopping

In [4]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0, path='best_model_trans.pth'):
        """
        Args:
            patience (int): Quante epoche aspettare dopo l'ultimo miglioramento prima di fermarsi.
            delta (float): Cambiamento minimo nella loss per essere considerato un vero miglioramento.
            path (str): Percorso dove salvare il modello migliore.
        """
        self.patience = patience
        self.delta = delta
        self.path = path
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss, model):
        # Se è la prima epoca, inizializza la best_loss
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        
        # Se la loss attuale non è scesa abbastanza rispetto alla migliore
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        
        # Se la loss è migliorata (è scesa)
        else:
            print(f"Validation loss diminuita ({self.best_loss:.4f} --> {val_loss:.4f}). Salvo il modello!")
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0 # Resetta il contatore

    def save_checkpoint(self, model):
        """Salva i pesi del modello quando la validation loss migliora."""
        torch.save(model.state_dict(), self.path)

## Useful Functions

In [5]:
def model_info(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Calcolo in MB (assumendo float32)
    size_in_mb = total_params * 4 / (1024 ** 2)
    
    # Uso i due punti e la virgola (:,) per mettere i separatori delle migliaia
    print(f"Parametri Totali:       {total_params:,}")
    print(f"Parametri Addestrabili: {trainable_params:,}")
    print(f"Dimensione (float32):   {size_in_mb:.2f} MB")
    
    return total_params, trainable_params, size_in_mb

def train_multitask_model(model, train_loader, val_loader, epochs= EPOCHS, learning_rate= LEARNING_RATE, device=None, weight_task= WEIGHT_TASK):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = model.to(device)

    early_stopping = EarlyStopping(patience=5, path='checkpoint_best_model_trans.pth')

    # Loss definition
    criterion_ai = nn.CrossEntropyLoss()
    criterion_domain = nn.CrossEntropyLoss()

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

    # 1. Crei lo scaler (una volta sola all'inizio del codice)
    scaler = amp.GradScaler('cuda')
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        running_train_loss = 0.0
        
        train_loop = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{epochs}] - Train")
        for images, labels in train_loop:
            # Move the samples on the devide (GPU or CPU)
            images = images.to(device)
            label_ai = labels['label_ai'].to(device)
            label_domain = labels['label_domain'].to(device)

            optimizer.zero_grad()

            # 2. Attiva l'autocast per il Forward Pass (usa Float16 dove possibile)
            with amp.autocast("cuda"):
                # Forward Pass: ottieni i logits dei task
                logits_binary, logits_transform = model(images)
                
                # Compute the loss functions
                loss_ai = criterion_ai(logits_binary, label_ai)
                loss_domain = criterion_domain(logits_transform, label_domain)

                # Combine the losses
                total_loss = loss_ai * weight_task + loss_domain * (1 - weight_task)

            # 3. Backward Pass e aggiornamento parametri ottimizzati con AMP
            # Invece di total_loss.backward(), scaliamo la loss per evitare l'underflow dei gradienti in Float16
            scaler.scale(total_loss).backward()
            
            # Invece di optimizer.step(), passiamo attraverso lo scaler
            scaler.step(optimizer)
            scaler.update()

            running_train_loss += total_loss.item()
            
            # Aggiorna la barra di avanzamento
            train_loop.set_postfix(loss=total_loss.item())
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        # --- VALIDATION PHASE ---
        model.eval()
        running_val_loss = 0.0
        correct_ai = 0
        correct_domain = 0
        total_samples = 0
        
        # Disabilita il calcolo dei gradienti per risparmiare memoria e velocizzare
        with torch.no_grad():
            val_loop = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{epochs}] - Val", leave=False)
            
            for images, labels in val_loop:
                images = images.to(device)
                label_ai = labels['label_ai'].to(device)
                label_domain = labels['label_domain'].to(device)

                with amp.autocast("cuda"):
                # Forward Pass
                    logits_binary, logits_transform = model(images)
                    
                    # Compute the loss functions
                    loss_ai = criterion_ai(logits_binary, label_ai)
                    loss_domain = criterion_domain(logits_transform, label_domain)

                    total_loss = loss_ai*weight_task + loss_domain*(1-weight_task)
                    running_val_loss += total_loss.item()
                
                # Accuracy: prendiamo l'indice con il valore massimo (argmax)
                _, preds_ai = torch.max(logits_binary, dim=1)
                _, preds_domain = torch.max(logits_transform, dim=1)
                
                total_samples += images.size(0)
                correct_ai += (preds_ai == label_ai).sum().item()
                correct_domain += (preds_domain == label_domain).sum().item()
                
        avg_val_loss = running_val_loss / len(val_loader)
        acc_ai = correct_ai / total_samples
        acc_domain = correct_domain / total_samples
        
        print(f"--- Epoch {epoch+1} ---")
        print(f"Training Loss:   {avg_train_loss:.4f}")
        print(f"Validation Loss: {avg_val_loss:.4f}")
        print(f"Val Accuracy AI: {acc_ai * 100:.2f}% | Val Accuracy Domain: {acc_domain * 100:.2f}%\n")

        # Early Stopping
        early_stopping(avg_val_loss, model)
        if early_stopping.early_stop:
            print("Early stopping triggered...")
            break
        
    print("Training Completed")
    return None

# Data

In [6]:
class RRDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_samples=MAX_SAMPLES):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        self.class_ai_to_idx = {"real": 0, "ai": 1}
        self.class_domain_to_idx = {"original": 0, "redigital": 1, "transfer": 2}

        for domain in self.class_domain_to_idx.keys():
            domain_path = self.root_dir / domain
            if not domain_path.exists(): continue
                
            for class_name in self.class_ai_to_idx.keys():
                class_path = domain_path / class_name
                for pattern in ("*.jpg", "*.png"):
                    for img_path in class_path.glob(pattern):
                        self.samples.append((img_path, self.class_ai_to_idx[class_name], self.class_domain_to_idx[domain]))

        
        # --- INIZIO CODICE DI BILANCIAMENTO ---
        if max_samples is not None and max_samples < len(self.samples):
            # A. Creiamo un dizionario per dividere le immagini nei 6 sottogruppi
            grouped_samples = defaultdict(list)
            for sample in self.samples:
                # sample[1] è label_ai, sample[2] è label_domain
                chiave_gruppo = (sample[1], sample[2]) 
                grouped_samples[chiave_gruppo].append(sample)
            
            # B. Calcoliamo quante immagini prendere per ogni "scatola"
            num_gruppi = len(grouped_samples) # Nel tuo caso sarà 6
            quota_per_gruppo = max_samples // num_gruppi
            
            balanced_samples = []
            random.seed(42) # Per riproducibilità
            
            # C. Estraiamo la quota esatta da ogni gruppo
            for chiave, immagini_gruppo in grouped_samples.items():
                # min() ci protegge nel raro caso in cui un gruppo abbia meno immagini della quota
                k = min(quota_per_gruppo, len(immagini_gruppo))
                balanced_samples.extend(random.sample(immagini_gruppo, k))
            
            # D. Mescoliamo tutto alla fine. (Fondamentale, altrimenti il modello
            # vedrebbe prima solo immagini reali originali, poi solo AI, ecc...)
            random.shuffle(balanced_samples)
            
            # Sostituiamo la lista intera con il nostro subset bilanciato
            self.samples = balanced_samples
        # --- FINE CODICE DI BILANCIAMENTO ---

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label_ai, label_domain = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return image, {
            'label_ai': torch.tensor(label_ai, dtype=torch.long),
            'label_domain': torch.tensor(label_domain, dtype=torch.long)
        }

my_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

dataset = RRDataset(root_dir=Path(".")/"RRDataset_test"/"RRDataset_final", transform=my_transforms)

In [7]:
dataset_size = len(dataset)
train_size = int(TRAIN_SIZE * dataset_size)
val_size = dataset_size - train_size

# Split
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=generator)

# DataLoaders
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    pin_memory=True,             # trasferimento CPU→GPU più veloce
)
val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False,
    pin_memory=True
)

print(f"Dataset Total size: {dataset_size}")
print(f"Training Set size: {len(train_dataset)}")
print(f"Validation Set size: {len(val_dataset)}")

Dataset Total size: 15000
Training Set size: 12000
Validation Set size: 3000


# Network

In [8]:
class PatchCreation(nn.Module):
    def __init__(self, input_color_channel: int, patch_size: int, embedding_dimensions : int):
        super().__init__()
        self.patch_size = patch_size
        # use Conv2d to simultaneously extract and project patches
        self.patching_conv = nn.Conv2d(
            input_color_channel,        # in_channels  = 3
            embedding_dimensions,       # out_channels = D = 256
            kernel_size=patch_size,     # 16×16 window
            stride=patch_size,          # non-overlapping
            padding=0
        )
        self.flatten = nn.Flatten(start_dim=2)

    def forward(self, x):
        # x: (B, C, H, W)
        image_dimension = x.shape[-1]
        assert image_dimension % self.patch_size == 0, \
            f"Given image dimension {image_dimension} is not divisible by patch size {self.patch_size}"
        if len(x.shape) == 3:  # allow input in (C, H, W) by adding batch dim
            x = x.unsqueeze(0)
        # output: (B, N, D)  where N = (H/P)*(W/P)
        return self.patching_conv(x).flatten(2).permute(0, 2, 1)

In [9]:
class ViTInputLayer(nn.Module):
    def __init__(self, in_channels: int,
                 patch_size: int,
                 image_size: int,
                 embedding_dimensions: int,
                 input_dropout_rate: float = 0.0):
        super().__init__()
        # Patch embedding module
        self.patch_embeddings = PatchCreation(in_channels,
                                              patch_size,
                                              embedding_dimensions)

        # Learnable CLS token
        # CLS token acts as a global summary of the image. After the encoder, its final state is used for classification.
        self.cls_token = nn.Parameter(
            torch.randn(1, 1, embedding_dimensions),
            requires_grad=True
        )

        # Number of patches
        num_patches = (image_size // patch_size) ** 2
        num_tokens = num_patches + 1  # +1 for CLS token

        # Learnable positional embeddings
        self.positional_embeddings = nn.Parameter(
            torch.randn(1, num_tokens, embedding_dimensions),
            requires_grad=True
        )

        # Optional dropout (helps regularization)
        self.dropout = nn.Dropout(p=input_dropout_rate)

    def forward(self, x):
        batch_size = x.shape[0]

        # Step 1: Get patch embeddings
        patch_embeddings = self.patch_embeddings(x)

        # Step 2: Expand CLS token for batch size
        cls_token = self.cls_token.expand(batch_size, -1, -1)

        # Step 3: Concatenate CLS token + patch embeddings
        tokens = torch.concat((cls_token, patch_embeddings), dim=1)

        # Step 4: Add positional embeddings + apply dropout
        return self.dropout(tokens + self.positional_embeddings)

In [10]:
class LayerNormalisation(nn.Module):
    def __init__(self, embed_dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(embed_dim)) # Scale factor
        self.bias = nn.Parameter(torch.zeros(embed_dim)) # Shift factor

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dimension: int, head: int, dropout_rate: float = 0.0):
        super().__init__()
        assert embedding_dimension % head == 0, "Embedding dimension must be divisible by number of heads."
        self.w_q = nn.Linear(embedding_dimension, embedding_dimension)
        self.w_k = nn.Linear(embedding_dimension, embedding_dimension)
        self.w_v = nn.Linear(embedding_dimension, embedding_dimension)
        self.head = head
        self.d_k = embedding_dimension // head
        self.w_o = nn.Linear(embedding_dimension, embedding_dimension)

        self.attention_dropout = nn.Dropout(p=dropout_rate)
        self.proj_dropout = nn.Dropout(p=dropout_rate)


    @staticmethod
    def attention(q, k, v, dropout: nn.Dropout = None):
        d_k = q.shape[-1]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
        scores = scores.softmax(dim=-1)
        if dropout is not None:
            scores = dropout(scores)
        return torch.matmul(scores, v), scores

    def forward(self, q, k, v):
        batch_size, num_tokens, _ = q.shape

        # 1. Linear projections + split into heads
        query = self.w_q(q).view(batch_size, num_tokens, self.head, self.d_k).transpose(1, 2)
        key = self.w_k(k).view(batch_size, num_tokens, self.head, self.d_k).transpose(1, 2)
        value = self.w_v(v).view(batch_size, num_tokens, self.head, self.d_k).transpose(1, 2)

        # 2. Scaled dot-product attention
        x, self.attention_score = MultiHeadAttention.attention(query, key, value, self.attention_dropout)

        # 3. Concatenate heads
        x = x.transpose(1, 2).contiguous().view(batch_size, num_tokens, self.head * self.d_k)

        # 4. Output projection
        return self.proj_dropout(self.w_o(x))

In [12]:
class FeedForwardLayer(nn.Module):
    # Applied token-wise after attention. It is a simple 2-layer MLP that increases then decreases dimensionality.
    def __init__(self, d_model: int, d_ff_scale: int = 2, dropout_rate: float = 0.1):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff_scale * d_model), # 256 → 1024
            nn.GELU(), # GELU (Gaussian Error Linear Unit) is smoother than ReLU and is used in most modern Transformers
            nn.Dropout(dropout_rate),
            nn.Linear(d_ff_scale * d_model, d_model), # 1024 → 256
            nn.Dropout(dropout_rate)
        )

    def forward(self, x):
        return self.mlp(x)

In [13]:
class EncoderBlock(nn.Module):
    def __init__(self, embedding_dimensions, heads, attention_dropout_rate, feed_forward_dropout_rate, dff_scale: int = 2):
        super().__init__()
        self.normalisation_stage1 = LayerNormalisation(embedding_dimensions)
        self.mhsa = MultiHeadAttention(embedding_dimensions, heads, attention_dropout_rate)
        self.normalisation_stage2 = LayerNormalisation(embedding_dimensions)
        self.feed_forward_layer = FeedForwardLayer(embedding_dimensions, d_ff_scale=dff_scale, dropout_rate=feed_forward_dropout_rate)

    def forward(self, x):
        residual1 = x
        x = self.normalisation_stage1(x)
        x = self.mhsa(x, x, x) + residual1
        residual2 = x
        return self.feed_forward_layer(self.normalisation_stage2(x)) + residual2

In [14]:
class Encoder(nn.Module):
    # Each encoder block processes the full sequence. After L=6 blocks, every token has "seen" every other token multiple times through the attention mechanism.
    def __init__(self, num_of_encoders, embeddings, dff_scale, heads, attention_dropout_rate, feed_forward_dropout_rate):
        super().__init__()
        self.encoder_stack = nn.ModuleList(
            EncoderBlock(
                embedding_dimensions=embeddings,
                heads=heads,
                dff_scale=dff_scale,
                attention_dropout_rate=attention_dropout_rate,
                feed_forward_dropout_rate=feed_forward_dropout_rate
            ) for _ in range(num_of_encoders)
        )

    def forward(self, x):
        for module in self.encoder_stack:
            x = module(x)
        return x

In [15]:
class ViT(nn.Module):
    def __init__(self, in_channels, image_size, patch_size, number_of_encoder, embeddings, d_ff_scale, heads, input_dropout_rate, attention_dropout_rate, feed_forward_dropout_rate):
        super().__init__()
        self.input_layer = ViTInputLayer(in_channels, patch_size, image_size, embeddings, input_dropout_rate)
        self.encoder_stack = Encoder(number_of_encoder, embeddings, d_ff_scale, heads, attention_dropout_rate, feed_forward_dropout_rate)

    def forward(self, x):
        x = self.input_layer(x)            # Patch embed + CLS + pos embed
        x = self.encoder_stack(x)          # L encoder blocks
        x = x[:, 0, :]                     # Extract CLS token: (B, C)
        return x # -> (B, D)

In [16]:
class BinaryClassifier(nn.Module):
    def __init__(self, embedding_dim: int = EMBEDDING_DIM, hidden_dim: int = 128, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        return self.net(x)


class TransformClassifier(nn.Module):
    def __init__(self, embedding_dim: int = EMBEDDING_DIM, hidden_dim: int = 128, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 3)
        )

    def forward(self, x):
        return self.net(x)


class MultiHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.binary_head = BinaryClassifier()
        self.transform_head = TransformClassifier()

    def forward(self, emb_fused):
        logits_binary = self.binary_head(emb_fused)
        logits_transform = self.transform_head(emb_fused)
        return logits_binary, logits_transform

In [17]:
class MultiTaskNet(nn.Module):
    def __init__(self):
        super().__init__()
        hyperparameters = {
            'in_channels': 3,
            'image_size': 224,
            'patch_size': 16,
            'number_of_encoder' : 4,
            'embeddings' : EMBEDDING_DIM,
            'd_ff_scale' : 2,
            'heads' : 4,
            'input_dropout_rate' : 0.1,
            'attention_dropout_rate' : 0.1,
            'feed_forward_dropout_rate' : 0.1,
        }
        self.backbone = ViT(**hyperparameters)
        self.multi_head = MultiHeadModel()

    def forward(self, x):
        x_emb = self.backbone(x)
        logits_binary, logits_transform = self.multi_head(x_emb)

        return logits_binary, logits_transform


In [18]:
model = MultiTaskNet()
model_info(model)

Parametri Totali:       2,423,429
Parametri Addestrabili: 2,423,429
Dimensione (float32):   9.24 MB


(2423429, 2423429, 9.244647979736328)

In [19]:
train_multitask_model(model, 
                      train_loader,
                      val_loader,
                      device= DEVICE)

Epoch [1/50] - Train: 100%|██████████| 60/60 [02:11<00:00,  2.20s/it, loss=0.856]


--- Epoch 1 ---
Training Loss:   0.8781
Validation Loss: 0.8528
Val Accuracy AI: 64.70% | Val Accuracy Domain: 38.90%



Epoch [2/50] - Train: 100%|██████████| 60/60 [02:12<00:00,  2.21s/it, loss=0.806]


--- Epoch 2 ---
Training Loss:   0.8406
Validation Loss: 0.7827
Val Accuracy AI: 74.47% | Val Accuracy Domain: 47.47%

Validation loss diminuita (0.8528 --> 0.7827). Salvo il modello!


Epoch [3/50] - Train: 100%|██████████| 60/60 [02:14<00:00,  2.25s/it, loss=0.742]


--- Epoch 3 ---
Training Loss:   0.7667
Validation Loss: 0.7253
Val Accuracy AI: 74.00% | Val Accuracy Domain: 50.13%

Validation loss diminuita (0.7827 --> 0.7253). Salvo il modello!


Epoch [4/50] - Train: 100%|██████████| 60/60 [02:11<00:00,  2.19s/it, loss=0.681]


--- Epoch 4 ---
Training Loss:   0.7111
Validation Loss: 0.6947
Val Accuracy AI: 73.77% | Val Accuracy Domain: 52.53%

Validation loss diminuita (0.7253 --> 0.6947). Salvo il modello!


Epoch [5/50] - Train:  17%|█▋        | 10/60 [00:22<01:54,  2.30s/it, loss=0.672]


KeyboardInterrupt: 